In [1]:
from pandas.core.arrays.integer import dtype
%load_ext autoreload
%autoreload 2

In [17]:
import itertools

from ast import literal_eval
from collections import Counter
from datetime import datetime


import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
#from sklearn.ensemble import ExtraTreesClassifier,
#from sklearn.preprocessing import LabelEncoder
#from sklearn.metrics import classification_report, f1_score

In [18]:
#from sherlock.deploy.model import SherlockModel

## Loading

In [19]:
# Check sherlock look
sherlock_labels = pd.read_parquet("../data/data/raw/train_labels.parquet").values.flatten()
print(sherlock_labels)


['area' 'collection' 'team Name' ... 'description' 'depth' 'product']


In [21]:
y_train = pd.read_parquet("./labels.parquet").values.flatten()
print(y_train)

['ID' 'ID' 'ID' 'Contact_ID' 'Age' 'Sex_at_birth' 'Case_status'
 'Date_onset' 'Date_report' 'Date_isolation' 'Date_hospitalisation'
 'Outcome' 'Pregnancy_Status' 'Healthcare_worker' 'Occupation'
 'Contact_with_case' 'Contact_setting' 'Vaccination' 'Vaccination_date'
 'Source' 'Date_last_modified' 'ID' 'Intensive_care' 'Genomics_Metadata'
 'Age' 'Case_status' 'Symptoms' 'Location_information'
 'Date_hospitalisation' 'Date_onset' 'Gender' 'Hospitalised' 'Outcome'
 'Location_information' 'Pre_existing_condition' 'Pre_existing_condition'
 'Previous_infection' 'Vaccine_name' 'Vaccination_date' 'Vaccination'
 'Pathogen' 'ID' 'Pathogen' 'ID' 'Case_status' 'Date_onset'
 'Date_confirmation' 'Confirmation_method' 'Date_report' 'Age'
 'Sex_at_birth' 'Outcome' 'Date_death' 'Hospitalised' 'Travel_history'
 'Pregnancy_Status' 'Symptoms' 'Race' 'Occupation' 'Occupation'
 'Pre_existing_condition' 'Pre_existing_condition'
 'Pre_existing_condition' 'Pre_existing_condition'
 'Pre_existing_condition' 'Pre

In [22]:
start = datetime.now()
print(f'Started at {start}')

X_train = pd.read_parquet("../data/data/processed/train-custom-data.parquet")

y_train = np.array([x.lower() for x in itertools.chain(y_train)])

print(f'Load data (train) process took {datetime.now() - start} seconds.')

Started at 2025-05-05 15:40:47.504350
Load data (train) process took 0:00:00.076159 seconds.


In [23]:
print('Distinct types for columns in the Dataframe (should be all float32):')
print(set(X_train.dtypes))

Distinct types for columns in the Dataframe (should be all float32):
{dtype('float32')}


In [24]:
print(X_train.shape)
print(y_train.shape)

(85, 1588)
(85,)


## Training

In [25]:
# n_estimators=300 gives a slightly better result (0.1%), but triples the fit time
voting_clf = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, random_state=13, n_jobs=-1)),
        #('et', ExtraTreesClassifier(n_estimators=100, #random_state=13, n_jobs=-1))
    ],
    voting='soft'
)

start = datetime.now()
print(f'Started at {start}')

voting_clf.fit(X_train, y_train)

print(f'Finished at {datetime.now()}, took {datetime.now() - start} seconds')

Started at 2025-05-05 15:41:13.488029
Finished at 2025-05-05 15:41:14.613939, took 0:00:01.125936 seconds


In [26]:
# Make individual (trained) estimators available
rf_clf = voting_clf.estimators_[0]
#et_clf = voting_clf.estimators_[1]

In [27]:
# 1) Ensemble predictions on training set
y_pred_ensemble = voting_clf.predict(X_train)

# 2) RandomForest‐only predictions on training set
y_pred_rf = rf_clf.predict(X_train)

In [28]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Ensemble accuracy
acc_ens = accuracy_score(y_train, y_pred_ensemble)
print(f"Ensemble training accuracy: {acc_ens:.4f}")

# RF‐only accuracy
acc_rf = accuracy_score(y_train, y_pred_rf)
print(f"RF‐only training accuracy:       {acc_rf:.4f}")

# Detailed report
print("\nEnsemble classification report:")
print(classification_report(y_train, y_pred_ensemble))

Ensemble training accuracy: 0.9294
RF‐only training accuracy:       0.0000

Ensemble classification report:
                        precision    recall  f1-score   support

                   age       1.00      1.00      1.00         3
           case_status       1.00      1.00      1.00         3
   confirmation_method       1.00      1.00      1.00         1
        contact_animal       1.00      1.00      1.00         1
            contact_id       1.00      1.00      1.00         1
       contact_setting       1.00      1.00      1.00         1
     contact_with_case       0.50      1.00      0.67         1
     date_confirmation       1.00      1.00      1.00         2
            date_death       1.00      1.00      1.00         1
  date_hospitalisation       1.00      1.00      1.00         2
        date_isolation       1.00      1.00      1.00         1
    date_last_modified       1.00      1.00      1.00         1
            date_onset       1.00      1.00      1.00      

/opt/miniconda3/envs/rosetta/lib/python3.7/site-packages/sklearn/metrics/_classification.py:208: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  score = y_true == y_pred
/opt/miniconda3/envs/rosetta/lib/python3.7/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/miniconda3/envs/rosetta/lib/python3.7/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/miniconda3/envs/rosetta/lib/python3.7/site-packages/sklearn/metrics/_classification.py:

## Prediction

In [34]:
from sherlock.features.paragraph_vectors import initialise_pretrained_model, initialise_nltk
from sherlock.features.word_embeddings import initialise_word_embeddings
from sherlock.features.preprocessing import prepare_feature_extraction

prepare_feature_extraction()
initialise_word_embeddings()
initialise_pretrained_model(400)
initialise_nltk()

Preparing feature extraction by downloading 4 files:
        
 ../sherlock/features/glove.6B.50d.txt, 
 ../sherlock/features/par_vec_trained_400.pkl.docvecs.vectors_docs.npy,
        
 ../sherlock/features/par_vec_trained_400.pkl.trainables.syn1neg.npy, and 
 ../sherlock/features/par_vec_trained_400.pkl.wv.vectors.npy.
        
All files for extracting word and paragraph embeddings are present.
Initialising word embeddings
Initialise Word Embeddings process took 0:00:01.978784 seconds.
Initialise Doc2Vec Model, 400 dim, process took 0:00:01.660683 seconds. (filename = ../sherlock/features/par_vec_trained_400.pkl)
Initialised NLTK, process took 0:00:00.132325 seconds.


[nltk_data] Downloading package punkt to /Users/omad/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/omad/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [35]:
data = pd.Series(
    [
        ["2024-02-30", "2020-03-01", "1982-12-30"],
        ["104805", "330956", "345609"],
        ["Male", "Female"],
        ["men", "women"],

    ],
    name="values"
)

In [36]:
data

0    [2024-02-30, 2020-03-01, 1982-12-30]
1                [104805, 330956, 345609]
2                          [Male, Female]
3                            [men, women]
Name: values, dtype: object

In [38]:
from sherlock.features.preprocessing import extract_features

extract_features(
    "../check.csv",
    data
)
feature_vectors = pd.read_csv("../check.csv", dtype=np.float32)

Extracting Features: 100%|██████████| 4/4 [00:00<00:00, 489.47it/s]

Exporting 1588 column features


In [39]:
feature_vectors

,n_[0]-agg-any,n_[0]-agg-all,n_[0]-agg-mean,n_[0]-agg-var,n_[0]-agg-min,n_[0]-agg-max,n_[0]-agg-median,n_[0]-agg-sum,n_[0]-agg-kurtosis,n_[0]-agg-skewness,...,par_vec_390,par_vec_391,par_vec_392,par_vec_393,par_vec_394,par_vec_395,par_vec_396,par_vec_397,par_vec_398,par_vec_399
0,1.0,1.0,2.666667,1.555556,1.0,4.0,3.0,8.0,-1.5,-0.381802,...,0.000482,-0.000699,0.000464,0.000695,0.000377,0.000414,-0.000623,-0.000379,-0.000771,-0.000442
1,1.0,1.0,1.333333,0.222222,1.0,2.0,1.0,4.0,-1.5,0.707107,...,0.000800,0.000959,0.000994,0.000532,0.000768,0.000870,-0.000838,-0.000695,0.000956,0.000615
2,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,-3.0,0.000000,...,-0.024608,-0.009890,-0.016155,0.000928,-0.008309,-0.015509,-0.020144,-0.037095,-0.001885,-0.040874
3,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,-3.0,0.000000,...,-0.072729,-0.022693,-0.110678,0.089974,-0.099107,-0.090429,-0.001951,-0.008750,0.021203,-0.082619


In [44]:
train_columns_means = pd.DataFrame(feature_vectors.mean()).transpose()
feature_vectors.fillna(train_columns_means.iloc[0], inplace=True)

In [45]:
model = rf_clf
#predicted_labels = model.predict(data)

# 1) Ensemble predictions on training set
y_pred_ensemble = voting_clf.predict(feature_vectors)

print(y_pred_ensemble)

# 2) RandomForest‐only predictions on training set
#y_pred_rf = rf_clf.predict(X_train)

['date_report' 'id' 'outcome' 'pre_existing_condition']
